In [1]:
import numpy as np
import os
import pickle
import tifffile as tiff

In [2]:
from nmf_utils import compute_dff

In [3]:
def main(file_movie, path_save, file_stem):
    """
    Extract average ΔF/F traces from final spatial components.

    This function:
    1. Loads the final (corrected) component masks from a saved bundle.
    2. Loads the calcium imaging movie as a memory-mapped array.
    3. Computes ΔF/F (dF/F) signals for each component individually.
    4. Averages the signal across all pixels within each component.
    5. Saves the resulting temporal traces to a new pickle file.

    Parameters
    ----------
    file_movie : str
        Path to the input TIFF movie (time x height x width).
    path_save : str
        Directory where the component bundle is stored and output will be saved.
    file_stem : str
        Base filename used to locate input and name output files.

    Returns
    -------
    None
        Results are saved to disk as:
        '{file_stem}_traces.pkl'

    Outputs (saved in bundle)
    -------------------------
    traces : np.ndarray
        Array of shape (N_components x T), where each row is the average
        ΔF/F trace for a component.
    parameters : dict
        Dictionary containing preprocessing parameters (e.g., percentile used).

    Notes
    -----
    - ΔF/F is computed per component using a percentile-based baseline.
    - Each component trace is obtained by averaging over its spatial mask.
    - Memory mapping is used to efficiently handle large movie files.
    """
    #Parameters
    percentile=10 # percentile to determine F0

    file_data =os.path.join(path_save, file_stem +"_final_components.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    final_components = bundle["final_components_corrected"]
    movie = tiff.memmap(file_movie)
    traces = []
    for comp in final_components:
        dff_masked, _ = compute_dff(movie, comp, percentile=percentile, return_full=False)
        traces.append(dff_masked.mean(axis=1))
    traces=np.array(traces)
    bundle = {        
    "traces": traces,
    "parameters":   {"percentile" :percentile}
    }
    # Save the bundle
    out_path = os.path.join(path_save, f"{file_stem}_traces.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    return

 

In [4]:
data_source = {
"exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT":["pup1_spont","pup2_spont"]
}

In [5]:
path_dist = "Data"
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        file_movie=os.path.join(path_dist, folder_exp, file_stem, file_stem +".tif")
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        main(file_movie,path_save,file_stem)
        print(f"Processing {file_stem} from {folder_exp}")


Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
